# LeRobot Config Compatibility with PhysicalAI

This notebook demonstrates **end-to-end config compatibility** between
[LeRobot](https://github.com/huggingface/lerobot) and PhysicalAI.

If you have an existing LeRobot `TrainPipelineConfig`, you can bring it
straight into the PhysicalAI training pipeline — no manual rewriting needed.

**What you'll learn:**

1. Create a LeRobot `TrainPipelineConfig` (as a LeRobot user would)
2. Detect config format automatically
3. Convert to PhysicalAI-native YAML via `TrainPipelineConfigAdapter`
4. Inspect the converted config
5. Use the CLI for conversion (`physicalai config --from lerobot`)
6. Train with the converted config (E2E)

### Prerequisites

- PhysicalAI library installed (`pip install -e library/`)
- LeRobot installed (`pip install lerobot`)
- PyYAML installed (`pip install pyyaml`)

## 1. Build a LeRobot TrainPipelineConfig

This is exactly how a LeRobot user creates their training config.
We use `lerobot/pusht` — a small 2-D push-T task that downloads
quickly and trains fast.

In [ ]:
from lerobot.configs.default import DatasetConfig, WandBConfig
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.configs.train import TrainPipelineConfig

# Build a standard LeRobot training config
lr_config = TrainPipelineConfig(
    policy=ACTConfig(),
    dataset=DatasetConfig(
        repo_id="lerobot/pusht",
        episodes=None,
    ),
    steps=100,
    batch_size=8,
    num_workers=0,
    log_freq=10,
    eval_freq=50,
    save_checkpoint=False,
    seed=1000,
    wandb=WandBConfig(enable=False),
)

print(f"Policy type : {lr_config.policy.type}")
print(f"Dataset     : {lr_config.dataset.repo_id}")
print(f"Steps       : {lr_config.steps}")
print(f"Batch size  : {lr_config.batch_size}")

## 2. Save the LeRobot Config to JSON

LeRobot configs are draccus dataclasses. We serialize to JSON — the
same format you'd get from a LeRobot training run's output directory.

In [ ]:
import draccus
import json
import tempfile
from pathlib import Path

tmpdir = Path(tempfile.mkdtemp())
lerobot_config_path = tmpdir / "lerobot_train_config.json"

# draccus.encode() correctly serializes LeRobot configs, including
# the polymorphic 'type' key on policy configs that dataclasses.asdict misses.
payload = draccus.encode(lr_config)
lerobot_config_path.write_text(json.dumps(payload, indent=2, default=str))

print(f"Saved to: {lerobot_config_path}")
print(f"File size: {lerobot_config_path.stat().st_size:,} bytes")
print()
print("Top-level keys:")
for k in payload:
    print(f"  - {k}")
print()
print(f"Policy type key present: {'type' in payload.get('policy', {})}")
print(f"Policy type value: {payload.get('policy', {}).get('type')}")

## 3. Auto-Detect Config Format

`detect_config_format()` inspects the top-level keys to determine
whether a file is a native PhysicalAI YAML or a LeRobot config.

In [ ]:
from physicalai.config.lerobot import detect_config_format

fmt = detect_config_format(lerobot_config_path)
print(f"Detected format: {fmt!r}")
assert fmt == "lerobot", f"Expected 'lerobot', got {fmt!r}"

## 4. Convert with TrainPipelineConfigAdapter

The adapter translates every LeRobot field into the
`model` / `data` / `trainer` structure that `physicalai fit --config`
expects. Fields with no PhysicalAI equivalent are preserved
under `_lerobot_extra`.

In [ ]:
from physicalai.config.lerobot import TrainPipelineConfigAdapter

adapter = TrainPipelineConfigAdapter(lr_config)
converted = adapter.to_dict()

print("Converted config top-level keys:")
for k in converted:
    print(f"  - {k}")

print()
print("--- model ---")
print(json.dumps(converted["model"], indent=2))

print()
print("--- data ---")
print(json.dumps(converted["data"], indent=2))

print()
print("--- trainer ---")
print(json.dumps(converted["trainer"], indent=2))

if "seed_everything" in converted:
    print()
    print(f"--- seed_everything: {converted['seed_everything']} ---")

## 5. Write the Converted Config as YAML

The adapter can write the converted config directly to a YAML file
that is ready for `physicalai fit --config`.

In [ ]:
native_yaml_path = tmpdir / "physicalai_train.yaml"
adapter.to_yaml(native_yaml_path)

print(f"Written to: {native_yaml_path}")
print()
print(native_yaml_path.read_text())

## 6. Verify the Converted YAML is Detected as Native

Quick sanity check: the generated YAML should be detected as `"physicalai"`.

In [ ]:
native_fmt = detect_config_format(native_yaml_path)
print(f"Detected format of generated YAML: {native_fmt!r}")
assert native_fmt == "physicalai", f"Expected 'physicalai', got {native_fmt!r}"

## 7. CLI Conversion: `physicalai config --from lerobot`

You can also convert from the command line without writing Python.
Here we invoke it via subprocess to show the exact CLI experience.

In [ ]:
import subprocess
import sys

cli_output_path = tmpdir / "cli_converted.yaml"

result = subprocess.run(
    [
        sys.executable, "-m", "physicalai.cli.cli",
        "config", "--from", "lerobot",
        str(lerobot_config_path),
        "-o", str(cli_output_path),
    ],
    capture_output=True,
    text=True,
)

print(f"Return code: {result.returncode}")
if result.stdout:
    print(f"stdout:\n{result.stdout}")
if result.stderr:
    print(f"stderr:\n{result.stderr}")

if cli_output_path.exists():
    print(f"\nCLI-generated YAML ({cli_output_path}):\n")
    print(cli_output_path.read_text())
else:
    print("\n(output file not created — check stderr above)")

assert result.returncode == 0, f"CLI failed with return code {result.returncode}:\n{result.stderr}"

## 8. Transparent Auto-Detection via `physicalai fit`

The PhysicalAI CLI can also accept a LeRobot config **directly**
with `physicalai fit --config lerobot_config.json`. The CLI
auto-detects the format and converts under the hood.

Below we replicate the same logic the CLI uses internally:
detect → adapt → write temp YAML → swap the config path.

In [ ]:
from physicalai.config.lerobot import detect_config_format, TrainPipelineConfigAdapter


def auto_convert_if_lerobot(args: list[str]) -> list[str]:
    """Replicate the CLI's auto-detection logic."""
    if "--config" not in args:
        return args
    idx = args.index("--config") + 1
    config_path = Path(args[idx])
    if detect_config_format(config_path) != "lerobot":
        return args
    # Convert on the fly
    lr_cfg = TrainPipelineConfig.from_pretrained(str(config_path))
    ad = TrainPipelineConfigAdapter(lr_cfg)
    tmp = tmpdir / "_auto_converted.yaml"
    ad.to_yaml(tmp)
    new_args = list(args)
    new_args[idx] = str(tmp)
    return new_args


# Simulate: physicalai fit --config lerobot_train_config.json
original_args = ["fit", "--config", str(lerobot_config_path)]
converted_args = auto_convert_if_lerobot(original_args)

print("Original args :")
print(f"  {original_args}")
print()
print("Converted args:")
print(f"  {converted_args}")
print()

config_idx = converted_args.index("--config") + 1
auto_yaml = Path(converted_args[config_idx])
print(f"Auto-generated YAML: {auto_yaml}")
print()
print(auto_yaml.read_text())

## 9. End-to-End Training

Now let's actually **train** using the original LeRobot config.
We construct the policy and data module from the converted dict
and run 5 training steps to prove the full pipeline works.

In [ ]:
from physicalai.policies.lerobot import ACT
from physicalai.data.lerobot import LeRobotDataModule, get_delta_timestamps_from_policy
from physicalai.train import Trainer

# Use the converted config values
model_cfg = converted["model"]
data_cfg = converted["data"]

fps = 10  # pusht recording frequency

# Build the policy from the converted model section
policy = ACT(
    n_obs_steps=1,
    chunk_size=100,
    optimizer_lr=1e-5,
)

# Build the data module from the converted data section
datamodule = LeRobotDataModule(
    repo_id=data_cfg["init_args"]["repo_id"],
    train_batch_size=data_cfg["init_args"]["train_batch_size"],
    data_format=data_cfg["init_args"]["data_format"],
    num_workers=data_cfg["init_args"]["num_workers"],
    delta_timestamps=get_delta_timestamps_from_policy("act", fps=fps),
    video_backend="pyav",
)

print(f"Policy     : {model_cfg['class_path']}")
print(f"Dataset    : {data_cfg['init_args']['repo_id']}")
print(f"Batch size : {data_cfg['init_args']['train_batch_size']}")

In [ ]:
# Create trainer and run 5 steps
trainer = Trainer(
    max_steps=5,
    accelerator="auto",
    enable_checkpointing=False,
    logger=False,
    enable_progress_bar=True,
)

trainer.fit(model=policy, datamodule=datamodule)
print("\nTraining complete!")

## 10. Inspect the Policy Class Map

The `POLICY_CLASS_MAP` is the canonical registry that maps LeRobot
policy names to PhysicalAI wrapper classes. Both the config adapter
and the policy factory use this single source of truth.

In [ ]:
from physicalai.policies.lerobot import POLICY_CLASS_MAP

print("Supported LeRobot policy -> PhysicalAI class mapping:\n")
for name, class_path in sorted(POLICY_CLASS_MAP.items()):
    print(f"  {name:12s} -> {class_path}")

## 11. Cleanup

In [ ]:
import shutil

shutil.rmtree(tmpdir, ignore_errors=True)
print(f"Cleaned up temp directory: {tmpdir}")

## Summary

| Step | What | How |
|------|------|-----|
| **Detect** | Identify config format | `detect_config_format(path)` |
| **Convert (Python)** | LeRobot config -> PhysicalAI YAML | `TrainPipelineConfigAdapter(cfg).to_yaml(path)` |
| **Convert (CLI)** | Same, from the command line | `physicalai config --from lerobot config.json -o out.yaml` |
| **Auto-detect (CLI)** | Pass LeRobot config directly | `physicalai fit --config lerobot_config.json` |
| **Train** | Use converted config to train | `Trainer().fit(model, datamodule)` |

### Key Points

- **Zero code changes** — LeRobot users bring their config as-is
- **Thin wrapper** — conversion happens at the config boundary only
- **Round-trip safe** — generated YAML is valid `physicalai fit` input
- **Lossless** — unmapped fields preserved in `_lerobot_extra`